In [3]:
!pip install scikit-learn

  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ---------------------------------------- 0.0/8.3 MB ? eta -:--:--
   ----------- ---------------------------- 2.4/8.3 MB 16.8 MB/s eta 0:00:01
   ------------------------------ --------- 6.3/8.3 MB 17.5 MB/s eta 0:00:01
   ---------------------------------------- 8.3/8.3 MB 17.7 MB/s  0:00:00
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)

   ---------- ----------------------------- 1/4 [narwhals]
   ---------- ----------------------------- 1/4 [narwhals]
   ---------- ----------------------------- 1/4 [narwhals]
   -------------------- ------------------- 2/4 [joblib]
   ------------------------------ --------- 3/4 [scikit-learn]
   ------------------------------ --------- 3/4 [scikit-learn]
   ------------------------------ --------- 3/4 [scikit-learn]
   ------------------------------ ---

In [7]:
import pandas as pd
import numpy as np
from sklearn.metrics import cohen_kappa_score
from pathlib import Path

CSV = Path(r"C:\Users\Vivek\Documents\dissertation\evaluation\spike_v2_scores.csv")
df = pd.read_csv(CSV)

axes = [
    "text_legibility",
    "regional_appropriateness", 
    "packaging_plausibility",
    "visual_quality",
]

print(f"Total scored rows: {len(df)}")
print()
print(f"{'Axis':<30} {'Kappa':>8} {'Mean S1':>8} {'Mean S2':>8} {'Δ Mean':>8} {'% Exact':>8} {'% ±1':>8}")
print("-" * 88)

results = {}
for axis in axes:
    s1 = pd.to_numeric(df[f"session1_{axis}"], errors="coerce")
    s2 = pd.to_numeric(df[f"session2_{axis}"], errors="coerce")
    mask = s1.notna() & s2.notna()
    s1v, s2v = s1[mask].astype(int), s2[mask].astype(int)
    
    # Cohen's weighted kappa (linear weights — the rubric's choice)
    kappa = cohen_kappa_score(s1v, s2v, weights="linear")
    
    mean1, mean2 = s1v.mean(), s2v.mean()
    diff = np.abs(s1v - s2v)
    pct_exact = (diff == 0).mean() * 100
    pct_within1 = (diff <= 1).mean() * 100
    
    print(f"{axis:<30} {kappa:>8.3f} {mean1:>8.2f} {mean2:>8.2f} {mean2-mean1:>+8.2f} {pct_exact:>7.1f}% {pct_within1:>7.1f}%")
    results[axis] = {"kappa": kappa, "n": len(s1v)}

print()
print("Cohen's kappa interpretation (Landis & Koch 1977):")
print("  < 0.00  Poor")
print("  0.00 - 0.20  Slight")  
print("  0.21 - 0.40  Fair")
print("  0.41 - 0.60  Moderate")
print("  0.61 - 0.80  Substantial")
print("  0.81 - 1.00  Almost perfect")

# Per-image difference distribution
print("\nPer-image total disagreement (sum of |S1-S2| across all 4 axes):")
total_diff = sum(
    np.abs(
        pd.to_numeric(df[f"session1_{ax}"], errors="coerce").fillna(0).astype(int) -
        pd.to_numeric(df[f"session2_{ax}"], errors="coerce").fillna(0).astype(int)
    )
    for ax in axes
)
print(f"  Mean total disagreement per image: {total_diff.mean():.2f}")
print(f"  Max disagreement: {total_diff.max()} (image: {df.loc[total_diff.idxmax(), 'filename']})")
print(f"  Rows with zero disagreement: {(total_diff == 0).sum()} ({(total_diff == 0).mean()*100:.1f}%)")

# Save the highest-disagreement images — these are worth a brief written note
high_disagree = df.loc[total_diff.sort_values(ascending=False).head(5).index, ["filename"] + 
                       [f"session{i}_{ax}" for i in [1,2] for ax in axes]]
high_disagree.to_csv(r"C:\Users\Vivek\Documents\dissertation\evaluation\spike_v2_high_disagreement.csv", index=False)
print(f"\nTop 5 most-disagreed images saved to evaluation/spike_v2_high_disagreement.csv")
print("Worth examining these — sometimes they reveal genuine rubric ambiguity.")

Total scored rows: 78

Axis                              Kappa  Mean S1  Mean S2   Δ Mean  % Exact     % ±1
----------------------------------------------------------------------------------------
text_legibility                   0.844     0.28     0.29    +0.01    93.6%   100.0%
regional_appropriateness          0.465     1.04     1.24    +0.21    61.5%   100.0%
packaging_plausibility            0.742     0.64     0.67    +0.03    79.5%   100.0%
visual_quality                    0.857     1.01     0.92    -0.09    88.5%   100.0%

Cohen's kappa interpretation (Landis & Koch 1977):
  < 0.00  Poor
  0.00 - 0.20  Slight
  0.21 - 0.40  Fair
  0.41 - 0.60  Moderate
  0.61 - 0.80  Substantial
  0.81 - 1.00  Almost perfect

Per-image total disagreement (sum of |S1-S2| across all 4 axes):
  Mean total disagreement per image: 0.77
  Max disagreement: 3 (image: baseline_p1_s1337.png)
  Rows with zero disagreement: 37 (47.4%)

Top 5 most-disagreed images saved to evaluation/spike_v2_high_disagre